# Msingi v10 — Swahili TTS with Split Acoustic + Vocoder Architecture

## Architectural decision: why split, and why BigVGAN

Earlier iterations trained Msingi as an end-to-end model with STFT-only
reconstruction loss. By ~64k steps this produced spectral collapse — the
model minimized L1/L2 on log-mel by predicting near-mean values, which sounds
like silence. This is the classic "autoencoder treatment" failure for audio
generation: mel-space reconstruction loss has near-zero correlation with
perceptual quality.

**The fix: separate the problem.**

1. **Acoustic model** (text → mel): Msingi v10, resumed from clean 64k.
   Trained with L1 + SSIM mel losses, no discriminator. The acoustic model's
   job is spectral prediction, not waveform realism.

2. **Vocoder** (mel → waveform): `nvidia/bigvgan_v2_22khz_80band_256x`,
   pre-trained, used as-is. BigVGAN uses Snake activation (periodic inductive
   bias) and anti-aliased up/downsampling — addresses the click artifacts
   that plagued HiFi-GAN-based attempts.

## Why BigVGAN over HiFi-GAN / training from scratch

- **Snake activation** `x + sin²(x)/α` — designed to represent periodic
  signals, which is what audio is. LeakyReLU can't.
- **Anti-aliased resampling** — removes metallic artifacts at layer boundaries.
- **Already multilingual (v2)** — trained on 100+ languages, generalizes to
  Swahili phonetics far better than a HiFi-GAN we'd train on 805 clips.
- **Pre-trained** — zero vocoder training cost.

## Contract: mel params must match

BigVGAN v2 22kHz expects:
- Sample rate 22050 Hz, 80 mel bands
- n_fft=1024, hop=256, win=1024
- fmin=0, fmax=8000, log-scale mels (natural log, not dB)

Msingi's mel extraction MUST match these exactly. Cell 4 (mel audit) verifies.


In [ ]:
import os
import modal

modal.enable_output()

CONFIG = {
    # Msingi checkpoint (clean 64k baseline)
    "checkpoint_resume": "/mnt/sauti/msingi_v10/ckpt_64000.pt",
    "output_dir": "/mnt/sauti/msingi_v10_revamp",

    # Mel params — MUST match BigVGAN v2 22kHz config
    "sample_rate": 22050,
    "n_fft": 1024,
    "hop_length": 256,
    "win_length": 1024,
    "n_mels": 80,
    "fmin": 0,
    "fmax": 8000,

    # Training
    "batch_size": 16,
    "learning_rate": 2e-4,
    "max_steps": 100_000,
    "grad_clip": 1.0,
    "warmup_steps": 1000,

    # Loss weights (VITS-style, no discriminator)
    "w_mel_l1": 45.0,
    "w_mel_ssim": 1.0,
    "w_duration": 1.0,
    "w_kl": 1.0,
    # NOTE: no w_disc, no w_gen, no w_fmaps — vocoder handles adversarial.

    # BigVGAN
    "vocoder_model": "nvidia/bigvgan_v2_22khz_80band_256x",

    # Logging
    "wandb_project": "sauti-ya-kenya",
    "wandb_run_name": "msingi-v10-revamp-acoustic",
    "log_every": 50,
    "save_every": 5000,
}

image = (
    modal.Image.debian_slim(python_version="3.12")
    .apt_install("ffmpeg", "git", "build-essential", "libsndfile1")
    .pip_install(
        "torch", "torchaudio",
        "transformers", "accelerate", "datasets",
        "soundfile", "librosa", "einops",
        "wandb", "pytorch-msssim",  # SSIM mel loss
    )
    # BigVGAN from source — small repo, compiled Snake CUDA kernels
    .run_commands(
        "cd /opt && git clone https://github.com/NVIDIA/BigVGAN.git",
        "cd /opt/BigVGAN && pip install -r requirements.txt",
    )
)

app = modal.App("msingi-v10-revamp")
vol = modal.Volume.from_name("sauti-tts-volume", create_if_missing=True)


## Mel extraction — the critical contract

Both the acoustic model and BigVGAN must produce/consume mels with identical
parameters. Mismatches here are the #1 cause of "vocoder produces noise"
bugs. This function is the single source of truth; use it everywhere.


In [ ]:
def mel_spectrogram(wav, config=None):
    """Canonical mel extraction matching BigVGAN v2 22kHz.

    Args:
        wav: (B, T) or (T,) float tensor, range [-1, 1]
        config: CONFIG dict (uses module-level CONFIG if None)

    Returns:
        (B, n_mels, T_frames) log-mel spectrogram, natural log.
    """
    import torch
    import torchaudio.transforms as T
    C = config or CONFIG

    if wav.dim() == 1:
        wav = wav.unsqueeze(0)

    # BigVGAN uses center=False padding; we match it
    mel_fn = T.MelSpectrogram(
        sample_rate=C["sample_rate"],
        n_fft=C["n_fft"],
        win_length=C["win_length"],
        hop_length=C["hop_length"],
        f_min=C["fmin"],
        f_max=C["fmax"],
        n_mels=C["n_mels"],
        power=1.0,        # magnitude, not power — BigVGAN convention
        center=False,
        pad_mode="reflect",
        norm="slaney",
        mel_scale="slaney",
    ).to(wav.device)

    mel = mel_fn(wav)
    # BigVGAN uses natural log with floor, not dB
    mel = torch.log(torch.clamp(mel, min=1e-5))
    return mel

print("Mel extractor defined. Parameters match nvidia/bigvgan_v2_22khz_80band_256x")


In [ ]:
def acoustic_losses(pred_mel, target_mel, pred_duration=None, target_duration=None, kl=None):
    """Compute acoustic model losses. No discriminator — vocoder handles that.

    Returns dict of individual losses and weighted total.
    """
    import torch
    import torch.nn.functional as F
    from pytorch_msssim import ssim

    C = CONFIG
    losses = {}

    # L1 on mel — sharp edges, not averaged-away predictions
    losses["mel_l1"] = F.l1_loss(pred_mel, target_mel)

    # SSIM on mel — structural similarity, captures formant/harmonic patterns
    # Treat mels as single-channel images: (B, 1, n_mels, T_frames)
    p = pred_mel.unsqueeze(1)
    t = target_mel.unsqueeze(1)
    # SSIM expects [0, 1] range; normalize with running min/max
    p_n = (p - p.min()) / (p.max() - p.min() + 1e-8)
    t_n = (t - t.min()) / (t.max() - t.min() + 1e-8)
    losses["mel_ssim"] = 1.0 - ssim(p_n, t_n, data_range=1.0, size_average=True)

    # Duration loss (if model predicts durations)
    if pred_duration is not None and target_duration is not None:
        losses["duration"] = F.l1_loss(
            torch.log(pred_duration + 1), torch.log(target_duration + 1)
        )

    # KL divergence (if VAE-style model)
    if kl is not None:
        losses["kl"] = kl

    # Weighted total
    total = (
        C["w_mel_l1"] * losses["mel_l1"]
        + C["w_mel_ssim"] * losses["mel_ssim"]
        + C["w_duration"] * losses.get("duration", 0.0)
        + C["w_kl"] * losses.get("kl", 0.0)
    )
    losses["total"] = total
    return losses

print("Loss function defined: L1 + SSIM + optional duration/KL. No adversarial.")


## Mel compatibility audit

Run this ONCE before training. Compares Msingi's mel extraction against
BigVGAN's expected input. Any mismatch here means the vocoder will produce
noise no matter how well the acoustic model trains.


In [ ]:
@app.function(
    image=image,
    gpu="RTX-PRO-6000",
    volumes={"/mnt/sauti": vol},
    timeout=600,
)
def audit_mel_compatibility():
    """Verify Msingi mel params match BigVGAN's expected input format."""
    import sys, torch, numpy as np
    sys.path.insert(0, "/opt/BigVGAN")

    from bigvgan import BigVGAN
    C = CONFIG

    print("[1/3] Loading BigVGAN")
    vocoder = BigVGAN.from_pretrained(C["vocoder_model"], use_cuda_kernel=False)
    vocoder.remove_weight_norm()
    vocoder.eval().cuda()

    print("[2/3] BigVGAN expected config:")
    vc = vocoder.h  # BigVGAN stores hyperparams in .h
    print("  sampling_rate:  %d (ours: %d)" % (vc.sampling_rate, C["sample_rate"]))
    print("  n_fft:          %d (ours: %d)" % (vc.n_fft, C["n_fft"]))
    print("  hop_size:       %d (ours: %d)" % (vc.hop_size, C["hop_length"]))
    print("  win_size:       %d (ours: %d)" % (vc.win_size, C["win_length"]))
    print("  num_mels:       %d (ours: %d)" % (vc.num_mels, C["n_mels"]))
    print("  fmin:           %d (ours: %d)" % (vc.fmin, C["fmin"]))
    print("  fmax:           %d (ours: %d)" % (vc.fmax, C["fmax"]))

    assert vc.sampling_rate == C["sample_rate"], "SR mismatch"
    assert vc.n_fft == C["n_fft"], "n_fft mismatch"
    assert vc.hop_size == C["hop_length"], "hop mismatch"
    assert vc.num_mels == C["n_mels"], "n_mels mismatch"

    print("[3/3] Round-trip test (wav → mel → vocoder → wav)")
    # Synthetic pure tone as sanity check
    t = torch.linspace(0, 2, C["sample_rate"] * 2)
    wav = 0.5 * torch.sin(2 * np.pi * 440 * t)  # A4 tone
    mel = mel_spectrogram(wav.cuda())
    with torch.no_grad():
        recon = vocoder(mel).squeeze().cpu().numpy()

    print("  Input RMS: %.4f | Recon RMS: %.4f" % (
        wav.pow(2).mean().sqrt().item(),
        float(np.sqrt(np.mean(recon ** 2))),
    ))
    print("  If recon RMS is near 0 or >> input RMS, mel params are wrong.")
    print("Audit complete.")


if __name__ == "__main__":
    with app.run():
        audit_mel_compatibility.remote()


## Acoustic model training

Resumes from clean 64k checkpoint. No discriminator, no GAN losses. Just
L1 + SSIM on mels with optional duration/KL from the model's own heads.

Expected behavior:
- mel_l1 decreases monotonically
- mel_ssim decreases slower but steadily (it's harder to fool)
- No training instability, no loss spikes (GAN oscillation gone)
- At ~85-100k, greedy mel predictions should contain clear formant structure


In [ ]:
@app.function(
    image=image,
    gpu="RTX-PRO-6000",
    volumes={"/mnt/sauti": vol},
    timeout=60 * 60 * 10,
    secrets=[modal.Secret.from_name("wandb-secret")],
)
def train_acoustic():
    """Resume Msingi v10 from 64k, train with L1+SSIM to 100k."""
    import os, torch, wandb
    from torch.optim import AdamW
    from torch.optim.lr_scheduler import LambdaLR

    C = CONFIG
    os.makedirs(C["output_dir"], exist_ok=True)

    # --- Model: your existing Msingi class ---
    # from msingi.model import Msingi  # adjust import
    # model = Msingi(C).cuda()
    # ckpt = torch.load(C["checkpoint_resume"], map_location="cuda")
    # model.load_state_dict(ckpt["model"])
    # start_step = ckpt.get("step", 64000)
    # TODO: replace placeholder below with your actual model import
    raise NotImplementedError(
        "Wire in your Msingi model class here. The training loop below is ready."
    )

    # --- Optimizer ---
    optim = AdamW(model.parameters(), lr=C["learning_rate"], betas=(0.8, 0.99))

    def lr_lambda(step):
        if step < C["warmup_steps"]:
            return step / C["warmup_steps"]
        return max(0.1, 1.0 - (step - C["warmup_steps"]) / (C["max_steps"] - C["warmup_steps"]))

    scheduler = LambdaLR(optim, lr_lambda)

    # --- W&B ---
    wandb.init(project=C["wandb_project"], name=C["wandb_run_name"], config=C)

    # --- Data loader: streaming, your existing one ---
    # train_loader = get_streaming_loader(...)  # TODO wire in
    # Placeholder for illustration:

    # --- Loop ---
    step = start_step
    model.train()
    while step < C["max_steps"]:
        for batch in train_loader:
            text_ids = batch["text_ids"].cuda()
            mel_target = batch["mel"].cuda()        # pre-computed or extracted inline
            durations = batch.get("durations")

            # Forward — your model returns (pred_mel, pred_duration, kl)
            pred_mel, pred_duration, kl = model(text_ids, mel_target, durations)

            losses = acoustic_losses(
                pred_mel, mel_target,
                pred_duration=pred_duration,
                target_duration=durations.cuda() if durations is not None else None,
                kl=kl,
            )

            optim.zero_grad()
            losses["total"].backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), C["grad_clip"])
            optim.step()
            scheduler.step()

            if step % C["log_every"] == 0:
                wandb.log({
                    "step": step,
                    "loss/total": losses["total"].item(),
                    "loss/mel_l1": losses["mel_l1"].item(),
                    "loss/mel_ssim": losses["mel_ssim"].item(),
                    "lr": scheduler.get_last_lr()[0],
                }, step=step)
                print("step %d | total %.4f | l1 %.4f | ssim %.4f" % (
                    step, losses["total"].item(),
                    losses["mel_l1"].item(), losses["mel_ssim"].item()))

            if step % C["save_every"] == 0 and step > start_step:
                path = os.path.join(C["output_dir"], "ckpt_%d.pt" % step)
                torch.save({"model": model.state_dict(), "step": step, "config": C}, path)
                vol.commit()
                print("Saved %s" % path)

            step += 1
            if step >= C["max_steps"]:
                break

    wandb.finish()
    print("Training complete at step %d" % step)


if __name__ == "__main__":
    with app.run():
        train_acoustic.remote()


## End-to-end synthesis: text → mel → audio

Full pipeline: Msingi acoustic model produces mel, BigVGAN converts to
waveform. This is what the research note evaluates.


In [ ]:
@app.function(
    image=image,
    gpu="RTX-PRO-6000",
    volumes={"/mnt/sauti": vol},
)
def synthesize(text, acoustic_ckpt=None):
    """Full TTS pipeline: text → mel (Msingi) → waveform (BigVGAN)."""
    import sys, torch, numpy as np
    sys.path.insert(0, "/opt/BigVGAN")
    from bigvgan import BigVGAN

    C = CONFIG
    ckpt_path = acoustic_ckpt or os.path.join(C["output_dir"], "ckpt_100000.pt")

    # --- Load acoustic model ---
    # from msingi.model import Msingi
    # model = Msingi(C).cuda().eval()
    # model.load_state_dict(torch.load(ckpt_path)["model"])
    # TODO: wire in Msingi
    raise NotImplementedError("Wire in Msingi inference here.")

    # --- Load BigVGAN ---
    vocoder = BigVGAN.from_pretrained(C["vocoder_model"], use_cuda_kernel=False)
    vocoder.remove_weight_norm()
    vocoder.eval().cuda()

    # --- Synthesize ---
    with torch.no_grad():
        text_ids = tokenize(text).cuda()               # your MsingiTokens tokenizer
        mel = model.inference(text_ids)                 # (1, n_mels, T_frames)
        waveform = vocoder(mel).squeeze().cpu().numpy()  # (T_audio,)

    return waveform.astype(np.float32)


from IPython.display import Audio as IPythonAudio, display

if __name__ == "__main__":
    with app.run():
        phrases = [
            "Mambo vipi? Ninafuraha sana kuzungumza nawe leo.",
            "Teknolojia ya akili bandia itabadilisha maisha yetu barani Afrika.",
            "Kuna wanafunzi 1,500 katika shule hii.",
        ]
        for i, text in enumerate(phrases):
            wav = synthesize.remote(text)
            print("Phrase %d (%.1f sec)" % (i + 1, len(wav) / CONFIG["sample_rate"]))
            display(IPythonAudio(wav, rate=CONFIG["sample_rate"]))


## Migration checklist from old notebook

Before deleting cells [11]-[24] from the old notebook:

- [ ] Confirm `/mnt/sauti/msingi_v10/ckpt_64000.pt` exists and is the clean
      pre-collapse checkpoint (not the 100k-step degraded one)
- [ ] Port your Msingi model class import into cell 5 (`train_acoustic`)
- [ ] Port your streaming data loader into cell 5
- [ ] Port your MsingiTokens tokenizer into cell 6 (`synthesize`)
- [ ] Run cell 4 (mel audit) FIRST — don't skip this. If it fails, don't train.
- [ ] Delete the discriminator classes you built in Phase 2. They're dead code
      now. The vocoder handles adversarial.
- [ ] Archive the old `train_msingi_v10_gan.py` somewhere if you want to
      reference it later, but don't import from it.

## What's explicitly gone

- Multi-Period Discriminator (MPD) — BigVGAN has its own
- Multi-Scale Discriminator (MSD) — BigVGAN has its own
- Generator vs discriminator balancing, LSGAN loss, feature matching loss —
  all moved to the vocoder, which is pre-trained
- STFT-only reconstruction — replaced by L1 + SSIM which preserves structure

## Monitoring during training

Watch these in W&B. If they look wrong, stop training:

- `loss/mel_l1` should drop from ~0.8 to ~0.3 over 30k steps
- `loss/mel_ssim` should drop from ~0.5 to ~0.2, slower
- No loss spikes (if you see them, grad_clip is too loose or LR too high)
- At step 70k, 80k, 90k — decode a sample through BigVGAN and listen. If it's
  still silent/clicky by 80k, something in the mel pipeline is wrong, not the
  training.

## References

- BigVGAN: https://github.com/NVIDIA/BigVGAN
- BigVGAN paper: Lee et al., "BigVGAN: A Universal Neural Vocoder with
  Large-Scale Training" (ICLR 2023)
- Snake activation: Ziyin et al., "Neural Networks Fail to Learn Periodic
  Functions and How to Fix It" (NeurIPS 2020)
